In [1]:
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("PyTorch installed:", end=" ")

try:
    import torch
    print(torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("No")

Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
Platform: Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
Working directory: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/notebooks
PyTorch installed: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import requests

api_url = "https://api.github.com/repos/mguti97/SoccerNet-v3D"

response = requests.get(api_url, timeout=20)
print("Repository status:", response.status_code)

if response.ok:
    repo = response.json()
    print("Repository:", repo["html_url"])
    print("Default branch:", repo["default_branch"])

    for endpoint in ["releases", "contents"]:
        url = f"{api_url}/{endpoint}"
        r = requests.get(url, timeout=20)
        print(f"\n{endpoint.upper()} status:", r.status_code)

        if r.ok:
            data = r.json()
            if endpoint == "releases":
                for release in data:
                    print("Release:", release["tag_name"])
                    for asset in release.get("assets", []):
                        print(" ", asset["name"], asset["browser_download_url"])
            else:
                for item in data:
                    print(item["name"], item["type"])
else:
    print(response.text[:500])

Repository status: 403
{"message":"API rate limit exceeded for 103.55.74.154. (But here's the good news: Authenticated requests get a higher rate limit. Check out the documentation for more details.)","documentation_url":"https://docs.github.com/rest/overview/resources-in-the-rest-api#rate-limiting"}



In [3]:
from pathlib import Path
import requests

model_dir = Path("models/soccernet_v3d")
model_dir.mkdir(parents=True, exist_ok=True)

url = (
    "https://github.com/mguti97/SoccerNet-v3D/"
    "releases/download/v1.0.0/yolo-sn-ball-opt.pt"
)
destination = model_dir / "yolo-sn-ball-opt.pt"

with requests.get(url, stream=True, timeout=60) as response:
    print("HTTP status:", response.status_code)
    response.raise_for_status()

    with destination.open("wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Saved to:", destination.resolve())
print("Size:", round(destination.stat().st_size / (1024**2), 2), "MB")

HTTP status: 200
Saved to: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/notebooks/models/soccernet_v3d/yolo-sn-ball-opt.pt
Size: 48.89 MB


In [4]:
from pathlib import Path
from ultralytics import YOLO
import torch

MODEL_PATH = Path("models/soccernet_v3d/yolo-sn-ball-opt.pt")

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH.resolve()}"

model = YOLO(str(MODEL_PATH))

print("Model loaded successfully")
print("Model task:", model.task)
print("Model classes:", model.names)
print("Model file size:", round(MODEL_PATH.stat().st_size / (1024**2), 2), "MB")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Model loaded successfully
Model task: detect
Model classes: {0: 'ball'}
Model file size: 48.89 MB
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [5]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

MODEL_PATH = Path("models/soccernet_v3d/yolo-sn-ball-opt.pt")
VAL_IMAGES = Path("../../../../soccernet_ball_yolo/soccernet_ball_yolo/images/val")

# Check the dataset path
print("Validation path exists:", VAL_IMAGES.exists())

if not VAL_IMAGES.exists():
    print("Set VAL_IMAGES to your local dataset's images/val directory.")
else:
    images = [
        p for p in VAL_IMAGES.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    print("Validation images found:", len(images))

    image_path = random.choice(images)
    model = YOLO(str(MODEL_PATH))

    results = model.predict(
        source=str(image_path),
        imgsz=960,
        conf=0.10,
        device=0,
        half=True,
        verbose=False
    )

    plotted = results[0].plot()
    plt.figure(figsize=(14, 8))
    plt.imshow(plotted[:, :, ::-1])
    plt.axis("off")
    plt.title(f"SoccerNet-v3D fine-tuned model: {image_path.name}")
    plt.show()

    print("Detections:", len(results[0].boxes))
    if len(results[0].boxes):
        print("Confidence scores:", results[0].boxes.conf.cpu().tolist())

Validation path exists: False
Set VAL_IMAGES to your local dataset's images/val directory.


In [7]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[2]
DATASET = PROJECT_ROOT / "datasets/football/processed/soccernet_ball_yolo"

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATASET.exists())

for p in [
    DATASET / "images/val",
    DATASET / "labels/val",
]:
    print(f"{p} -> exists={p.exists()}")

Project root: /mnt/d/My Projects/AthletiQ
Dataset exists: False
/mnt/d/My Projects/AthletiQ/datasets/football/processed/soccernet_ball_yolo/images/val -> exists=False
/mnt/d/My Projects/AthletiQ/datasets/football/processed/soccernet_ball_yolo/labels/val -> exists=False


In [8]:
from pathlib import Path

PROJECT_ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
DATASET = PROJECT_ROOT / "datasets/football/processed/soccernet_ball_yolo"

VAL_IMAGES = DATASET / "images/val"
VAL_LABELS = DATASET / "labels/val"

print("Dataset exists:", DATASET.exists())
print("Validation images exist:", VAL_IMAGES.exists())
print("Validation labels exist:", VAL_LABELS.exists())

if VAL_IMAGES.exists():
    print("Validation images:", sum(
        1 for p in VAL_IMAGES.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ))

if VAL_LABELS.exists():
    print("Validation label files:", len(list(VAL_LABELS.glob("*.txt"))))

Dataset exists: True
Validation images exist: True
Validation labels exist: True
Validation images: 11250
Validation label files: 11250
